<a href="https://colab.research.google.com/github/FranciscoBPereira/NTC_IA-2627/blob/main/NTCIA2627_Aula1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42


**0. Leitura do Dataset para uma dataframe Pandas**

In [ ]:
# Leitura do dataset. O ficheiro 'ObesityDataset.csv' deve estar colocado na diretoria de trabalho

# Esta é uma versão corrigida do dataset original que está disponível aqui: https://www.kaggle.com/datasets/ezzaldeenesmail/obesitydataset-raw-and-data-sinthetic
# 1. Foram eliminadas entradas duplicadas
# 2. A parte decimal dos atributos numéricos foi eliminada

# O dataset fica armazenado numa dataframe Pandas chamada df

df= pd.read_csv('ObesityDataset.csv')

# Dimensão do dataset:
# Número de Exemplos / Atributos

print('Dimensão: ', df.shape)

print('Número de exemplos: ', df.shape[0])
print('Número de atributos: ', df.shape[1])

# Alterar o nome da coluna Target

df = df.rename(columns={'NObeyesdad': 'Obesity'})

df.info()


**1. Transformar Dados**

Todos os atributos categóricos passam a numéricos (ver aula anterior)

In [ ]:
# Existe inconsistência na forma de representar e identificar atributos categóricos
# Alguns estão identificados como numéricos e outros como objetos

# Aplica-se uma transformação e todos passam a ser considerados numéricos, o que vai simplificar a análise

ordem_obesity = ["Insufficient_Weight", "Normal_Weight",
         "Overweight_Level_I", "Overweight_Level_II",
         "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"]

# Mapas de codificação (valor original → código)
bin_map    = {'no': 0, 'yes': 1}
gender_map = {'Female': 0, 'Male': 1}
freq_map   = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
obesity_map = {k: i for i, k in enumerate(ordem_obesity)}

# Mapas inversos (código → valor original), para gráficos e interpretação
bin_inv     = {v: k for k, v in bin_map.items()}
gender_inv  = {v: k for k, v in gender_map.items()}
freq_inv    = {v: k for k, v in freq_map.items()}
obesity_inv = {v: k for k, v in obesity_map.items()}

# Aplicar

# Mapear os atributos binários:
for c in ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']:
    df[c] = df[c].map(bin_map)

df['Gender'] = df['Gender'].map(gender_map)

# Mapear os atributos com 4 possibilidades: No; Sometimes; Frequently; Always
for c in ['CAEC', 'CALC']:
    df[c] = df[c].map(freq_map)

# Mapear o target
df['Obesity'] = df['Obesity'].map(obesity_map)

# Mapear o atributo categórico não ordinal (codificação one-hot)
df = pd.get_dummies(df, columns=['MTRANS'], dtype=int)

df.info()


**2. Aprendizagem Supervisionada - Experiências Iniciais**

Vão ser testados três algoritmos supervisionados: Decision Tree, Random Forest e MLP (Redes Neuronais)

Nesta fase serão usados todos os atributos sem nenhuma transformação adicional

**2.1. Preparação do dataset**

In [ ]:
# Separar atributos (variáveis independentes) do target (classe dos exemplos - variável dependente)
# Isto separa o input dos modelos do resultado esperado

Att = df.drop(['Obesity'], axis=1)
Classe = df['Obesity']

print('Target: ', Classe.shape)
print('Atributos: ', Att.shape)

In [ ]:
# Criar conjuntos de treino (75% dos exemplos) e teste (25% dos exemplos)

# Conjunto de treino: {Att_treino, Classe_treino}
# Conjunto de teste: {Att_teste, Classe_teste}

# A escolha dos exemplos é efetuada de forma aleatória, mantendo a proporção original dos exemplos de cada categoria

from sklearn.model_selection import train_test_split

Att_treino, Att_teste, Classe_treino, Classe_teste = train_test_split(
    Att, Classe, test_size=0.25, stratify=Classe, random_state = SEED)

print('Atributos de treino: ', Att_treino.shape)
print('Atributos de teste: ', Att_teste.shape)
print('Target de treino: ', Classe_treino.shape)
print('Target de teste: ', Classe_teste.shape)


In [ ]:
# Função auxiliar para mostrar uma matriz de confusão com um formato gráfico

from sklearn.utils.multiclass import unique_labels

def plot_confusion_matrix(y_true, y_pred, classes, display_labels=None,
                          normalize=False, title=None, cmap=plt.cm.Blues):
    if not title:
        title = 'Normalized confusion matrix' if normalize else 'Confusion matrix, without normalization'

    cm = confusion_matrix(y_true, y_pred, labels=classes)
    names = display_labels if display_labels is not None else classes
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    sns.set(font_scale=1.2)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
           xticklabels=names, yticklabels=names, title=title,
           ylabel='True label', xlabel='Predicted label')
    ax.grid(False)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax


**2.2. Treinar e Avaliar uma Árvore de Decisão**

In [ ]:
#from sklearn.metrics import recall_score, precision_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import plot_tree

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


In [ ]:
# Criar um modelo árvore de decisão
tree = DecisionTreeClassifier(random_state = SEED)

# Treinar o modelo árvore com os dados de treino
tree.fit(Att_treino, Classe_treino)



In [ ]:
# Mostrar resumo dos resultados da árvore de decisão

print(classification_report(Classe_teste, tree.predict(Att_teste),target_names=ordem_obesity))

In [ ]:
# Mostrar resultados da árvore de decisão no formato matriz de confusão

y_pred = tree.predict(Att_teste)

plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

In [ ]:
# Consultar os primeiros três níveis da árvore

fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(tree, max_depth=3,
          feature_names=tree.feature_names_in_,
          class_names=ordem_obesity,
          filled=True, rounded=True,
          impurity=False, proportion=True,
          label='root', precision=2, fontsize=10, ax=ax)

# Guardar a árvore num ficheiro
fig.savefig('arvore.png', dpi=300, bbox_inches='tight')   # imagem
plt.show()



**2.3. Treinar e Avaliar uma Random Forest**

In [ ]:

# Criar um modelo random forest
rf = RandomForestClassifier(random_state = SEED)

# Treinar um modelo random forest
rf.fit(Att_treino, Classe_treino)



In [ ]:
# Mostrar resumo dos resultados Random Forest

print(classification_report(Classe_teste, rf.predict(Att_teste),target_names=ordem_obesity))

In [ ]:
# Mostrar resultados da Random Forest no formato matriz de confusão

y_pred = rf.predict(Att_teste)

plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

**2.4. Treinar e Avaliar uma Rede Neuronal (MLP)**

In [ ]:
# Criar um modelo MLP
mlp = MLPClassifier(max_iter=1000, random_state = SEED)

# Treinar um modelo MLP
mlp.fit(Att_treino, Classe_treino)


In [ ]:
# Mostrar resumos dos resultados MLP

print(classification_report(Classe_teste, mlp.predict(Att_teste),target_names=ordem_obesity))

In [ ]:
# Mostrar resultados da MLP no formato matriz de confusão

y_pred = mlp.predict(Att_teste)

plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

**3. Standardização dos Atributos para Classificadores Paramétricos**

Os resultados da rede neuronal são fracos, quando comparados com os dos outros 2 métodos

A standardização de valores dos atributos ajuda a melhorar o desempenho

In [ ]:
# A MLP é um método paramétrico
# Existe vantagem se os dados estiverem normalizados / standardizados

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Att_treino_s = scaler.fit_transform(Att_treino)   # Calcula média/desvio no treino e transforma
Att_teste_s  = scaler.transform(Att_teste)        # Aplica os mesmos valores ao teste

# Criar nova MLP
mlpSTD = MLPClassifier(max_iter=1000, random_state = SEED)

# Treinar a nova MLP com os dados standardizados
mlpSTD.fit(Att_treino_s, Classe_treino)

# Verificar os resultados
print(classification_report(Classe_teste, mlpSTD.predict(Att_teste_s),
                            target_names=ordem_obesity))

In [ ]:
y_pred = mlpSTD.predict(Att_teste_s)

plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

**4. Aprendizagem Supervisionada - Sem Peso e Altura**

Estudo do impacto do peso e da altura (descritores que definem diretamente a classe)

Estudo efetuado apenas para árvores de decisão.
Fora da aula deverá ser expandido para os restantes algoritmos de classificação usados

In [ ]:
# Criar um novo dataset eliminando as colunas peso e altura do dataset
# Está a ser usada a versão sem standardização. Se pretender treinar uma MLP deve adicionar esse passo

Att_treino_sem = Att_treino.drop(columns=['Height', 'Weight'])
Att_teste_sem  = Att_teste.drop(columns=['Height', 'Weight'])


In [ ]:
# Criar, treinar e avaliar uma árvore de decisão que não usa os atributos peso e altura

treeR1 = DecisionTreeClassifier(random_state = SEED)
treeR1.fit(Att_treino_sem, Classe_treino)

print(classification_report(Classe_teste, treeR1.predict(Att_teste_sem),target_names=ordem_obesity))

y_pred = treeR1.predict(Att_teste_sem)

plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

In [ ]:
# Consultar os primeiros três níveis da árvore

fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(treeR1, max_depth=3,
          feature_names=treeR1.feature_names_in_,
          class_names=ordem_obesity,
          filled=True, rounded=True,
          impurity=False, proportion=True,
          label='root', precision=2, fontsize=10, ax=ax)

# Guardar a árvore num ficheiro
fig.savefig('arvore_R1.png', dpi=300, bbox_inches='tight')   # imagem
plt.show()

**5. Aprendizagem Supervisionada - Determinar Conjunto de Atributos mais relevantes**

Se existir um limite para o número de atributos a considerar, quais são os mais relevantes?

In [ ]:
# São definidos 3 grupos: Alimentação, Atividade e Demografia
# Qual é o menos relevante?

def colunas(df_att, variaveis):
    return [c for c in df_att.columns
            if any(c == v or c.startswith(v + '_') for v in variaveis)]

# Definição dos Grupos
grupos = {
    'Alimentação': ['FAVC', 'FCVC', 'NCP', 'CAEC', 'CH2O', 'SCC', 'CALC'],
    'Atividade':   ['FAF', 'TUE', 'MTRANS', 'SMOKE'],
    'Demografia':  ['Gender', 'Age', 'family_history_with_overweight'],
}

grupos['G1'] = grupos['Alimentação'] + grupos['Atividade']
grupos['G2'] = grupos['Alimentação'] + grupos['Demografia']
grupos['G3'] = grupos['Atividade'] + grupos['Demografia']




Questão: Dos 3 grupos criados qual é o menos relevante para o desempenho do classificador?

In [ ]:
# Não considerar demografia

Select = 'G1'

# Selecionar as colunas relevantes
cols = colunas(Att_treino_sem, grupos[Select])
Treino_Sel = Att_treino_sem[cols]
Teste_Sel  = Att_teste_sem[cols]

# Criar, treinar e avaliar a árvore

T1 = DecisionTreeClassifier(random_state = SEED)
T1.fit(Treino_Sel, Classe_treino)

y_pred = T1.predict(Teste_Sel)

print(classification_report(Classe_teste, y_pred,target_names=ordem_obesity))


plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

In [ ]:
# Não considerar atividade

Select = 'G2'

# Selecionar as colunas relevantes
cols = colunas(Att_treino_sem, grupos[Select])
Treino_Sel = Att_treino_sem[cols]
Teste_Sel  = Att_teste_sem[cols]

# Criar, treinar e avaliar a árvore

T2 = DecisionTreeClassifier(random_state = SEED)
T2.fit(Treino_Sel, Classe_treino)

y_pred = T2.predict(Teste_Sel)

print(classification_report(Classe_teste, y_pred,target_names=ordem_obesity))


plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()

In [ ]:
# Não considerar alimentação

Select = 'G3'

# Selecionar as colunas relevantes
cols = colunas(Att_treino_sem, grupos[Select])
Treino_Sel = Att_treino_sem[cols]
Teste_Sel  = Att_teste_sem[cols]

# Criar, treinar e avaliar a árvore

T3 = DecisionTreeClassifier(random_state = SEED)
T3.fit(Treino_Sel, Classe_treino)

y_pred = T3.predict(Teste_Sel)

print(classification_report(Classe_teste, y_pred,target_names=ordem_obesity))


plot_confusion_matrix(Classe_teste, y_pred,
                      classes=list(range(len(ordem_obesity))),
                      display_labels=ordem_obesity,
                      normalize=True)

plt.show()